# Niedopasowanie modelu i margines wyboru wariantu (osi c)

Dotychczasowe testy generowały dane **tym samym** modelem, którym dopasowujemy — scenariusz
najbardziej optymistyczny. Tu dane generuje "prawda" różniąca się od modelu, a dopasowanie
zawsze robi `ConeModel(θ = 40°, σ_conv = 0)` na pełnym masterze. Pytania:

1. **Jak niedopasowanie wpływa na orientację i wybór wariantu?** Obciążenie orientacji
   (bias) w poprawnym wariancie oraz odsetek poprawnie wybranych wariantów (oś c, polarność).
2. **Czy margines decyzji jest wiarygodnym wskaźnikiem?** Na prawdziwych danych nie ma
   etykiet, więc jedyne, co widać, to Δr = r_najlepszy − r_drugi (i ΔlogL). Sprawdzamy,
   przy jakim Δr decyzja jest pewna — *we wszystkich* scenariuszach niedopasowania.
3. **Czy łączenie pikseli domeny pomaga?** Szum się uśrednia, ale niedopasowanie
   strukturalne jest w każdym pikselu takie samo — może się kumulować.

**Nowe mastery nie są potrzebne** dla: θ, rozmycia wiązki, bicia stolika, nieliniowości
detektora i (w przybliżeniu) wag pozycji atomów. **Są potrzebne** dla: energii wiązki, c/a,
przesunięć jonów (polaryzacji), poprawnie policzonych DW, absorpcji — wtedy podaj gotowy
plik jako `MASTER_TRUTH`.

Tryb `"fast"`: optymalizacja startuje z 6 wariantów prawdziwej orientacji (bez szukania
globalnego — w testach A/B szukanie nie zawodziło, a warianty i tak są sprawdzane jawnie).
Tryb `"full"`: pełna procedura `fit_profile(..., score_variants=True)`, ~5× wolniej.

## 0. Parametry

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import echord_cone as ec
import orient_fit as of
import mismatch as mm

MASTER       = "../2026_10_19_example_PZ/masterKTN_014_01.h5"
MASTER_TRUTH = None              # inny master (np. inna energia / c/a) jako prawda, albo None

THETA, INVERT = 40.0, False      # model dopasowania
PHI0, SENSE   = 13.0, -1
N_TEST  = 60                     # pikseli na scenariusz
SNR     = 10                     # szum w KAŻDYM scenariuszu (std profilu / std szumu)
GAIN_SD = 0.3
MODE    = "fast"                 # "fast" albo "full"
SPACING, TOP_K, N_REFINE = 0.7, 5, 6      # tylko dla MODE = "full"
BANK_CACHE = "bank"
SEED = 0

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Master, model dopasowania, kanały mastera

In [ ]:
mp = ec.MasterPattern(MASTER)
print(mp)
fit = of.ConeModel(mp, THETA, INVERT)
VARIANTS = of.pseudo_variants()
D4, N_OMEGA = of.d4_ops(), len(fit.omega)

ch = pd.DataFrame(mm.channel_contributions(mp, MASTER)).set_index("kanal")
print(f"\nkanały mastera: {len(ch)} (bez osi energii -> najpewniej pozycje atomów, sumowane)")
display(ch.style.format({"srednia": "{:.4g}", "std": "{:.4g}", "udzial_sredniej": "{:.1%}",
                         "r_z_suma": "{:.4f}"}))

bank = None
if MODE == "full":
    path = f"{BANK_CACHE}_cone_sp{SPACING}_bg0.npz" if BANK_CACHE else None
    bank = of.Bank.load(path) if path and os.path.exists(path) else of.build_bank(fit, SPACING)
    if path and not os.path.exists(path):
        bank.save(path)

## 2. Funkcje pomocnicze

Dla każdego piksela: 6 wariantów dopracowanych osobno → wektor r (6). Z niego:

* **bias** — dezorientacja wariantu prawdziwego (po dopracowaniu) względem prawdy:
  obciążenie orientacji wywołane niedopasowaniem, nawet gdy wariant jest dobry,
* **poprawny** — czy wygrał prawdziwy wariant; jeśli nie, czy to pomyłka polarności, czy osi c,
* **dr_obs, dlogL_obs** — margines widoczny bez etykiet (to samo policzysz na pomiarze),
* **dr_true** = r_prawdziwy − max r_pozostałych (ujemny = zła decyzja).

In [ ]:
g_true = of.random_orientations(N_TEST, SEED)
results, SCEN = {}, {}


def perturb(Y, seed):
    rng = np.random.default_rng(seed)
    s = Y.std(axis=1, keepdims=True)
    out = (1 + GAIN_SD * rng.normal(size=(len(Y), 1))) * Y + s * rng.normal(size=(len(Y), 1))
    return out + rng.normal(size=Y.shape) * s / SNR if np.isfinite(SNR) else out


def analyze(y, gt):
    if MODE == "fast":
        G, R = of.variant_scores(fit, y, gt, VARIANTS, PHI0, SENSE)
    else:
        res = of.fit_profile(fit, bank, y, PHI0, SENSE, top_k=TOP_K, variants=VARIANTS,
                             n_refine=N_REFINE, score_variants=True)
        G, R = res["variant_g"], res["variant_r"]
    mis = of.misorientation_deg(G, np.repeat(gt[None], len(G), 0))
    it, ib = int(np.argmin(mis)), int(np.argmax(R))
    dr_obs, dl_obs = of.margin_stats(R, N_OMEGA)
    wrong = "" if it == ib else (
        "polarność" if of.misorientation_deg(G[ib], gt, D4)[0] < 2 else "oś c (90°)")
    return dict(poprawny=(it == ib) and mis[it] < 2, blad=wrong, bias=mis[it],
                szukanie_zawiodlo=mis[it] > 2, r_best=R[ib], r_truevar=R[it],
                dr_obs=dr_obs, dlogL_obs=dl_obs,
                dr_true=R[it] - np.max(np.delete(R, it)), R=R)


def run(name, truth, transform=None, seed=1):
    # truth: obiekt z .profiles(g, phi0, sense); transform: funkcja na profilach (np. gamma)
    t0 = time.perf_counter()
    Y = truth.profiles(g_true, PHI0, SENSE)
    if transform is not None:
        Y = transform(Y)
    Y = perturb(Y, seed)
    rows = []
    for i, y in enumerate(Y):
        rows.append(analyze(y, g_true[i]))
    df = pd.DataFrame(rows)
    results[name], SCEN[name] = df, dict(truth=truth, transform=transform, Y=Y)
    print(f"[{name}] {(time.perf_counter() - t0) / len(Y):.2f} s/piksel, "
          f"poprawny wariant {df.poprawny.mean():.0%}, mediana biasu {df.bias.median():.3f}°")
    return df


def summary(names=None):
    names = list(results) if names is None else names
    out = {}
    for k in names:
        d = results[k]
        out[k] = {"poprawny wariant": d.poprawny.mean(),
                  "błąd: oś c": (d.blad == "oś c (90°)").sum(),
                  "błąd: polarność": (d.blad == "polarność").sum(),
                  "szukanie": d.szukanie_zawiodlo.sum(),
                  "bias med [°]": d.bias.median(), "bias p90 [°]": d.bias.quantile(0.9),
                  "r_best med": d.r_best.median(), "Δr_obs med": d.dr_obs.median(),
                  "ΔlogL_obs med": d.dlogL_obs.median()}
    return (pd.DataFrame(out).T.style.format({
        "poprawny wariant": "{:.0%}", "błąd: oś c": "{:.0f}", "błąd: polarność": "{:.0f}",
        "szukanie": "{:.0f}", "bias med [°]": "{:.3f}", "bias p90 [°]": "{:.3f}",
        "r_best med": "{:.4f}", "Δr_obs med": "{:.4f}", "ΔlogL_obs med": "{:.1f}"}))

## 3. Odniesienie: brak niedopasowania (tylko szum)

In [ ]:
run("ref: bez niedopasowania", fit)
summary()

## 4. Scenariusze niedopasowania

### 4a. Inne θ (kalibracja kąta pochylenia)
Z poprzednich obserwacji: 39° / 41° zmieniają profile bardzo mocno.

In [ ]:
DTHETA = [-1.0, -0.5, -0.2, 0.2, 0.5, 1.0]
for dt in DTHETA:
    run(f"θ {dt:+.1f}°", of.ConeModel(mp, THETA + dt, INVERT))
summary(["ref: bez niedopasowania"] + [f"θ {dt:+.1f}°" for dt in DTHETA])

### 4b. Rozmycie wiązki (splot po stożku, σ_conv)

In [ ]:
SIGMAS = [0.1, 0.25, 0.5, 1.0]
for s in SIGMAS:
    run(f"σ_conv {s}°", of.ConeModel(mp, THETA, INVERT, sigma_conv_deg=s))
summary([f"σ_conv {s}°" for s in SIGMAS])

### 4c. Bicie stolika: θ(ω) = θ₀ + ε·cos(ω + α)
Oś obrotu niedokładnie prostopadła do wiązki / precesja — ścieżka nie jest okręgiem.

In [ ]:
EPS = [0.1, 0.25, 0.5]
for e in EPS:
    run(f"bicie ε {e}°", mm.PathModel(mp, mm.wobble(THETA, e, alpha=30.0), INVERT))
summary([f"bicie ε {e}°" for e in EPS])

### 4d. Nieliniowość detektora: I → I^γ

In [ ]:
GAMMAS = [0.7, 0.85, 1.15, 1.3]
for gm in GAMMAS:
    run(f"γ {gm}", fit, transform=lambda Y, gm=gm: mm.nonlinear(Y, gm))
summary([f"γ {gm}" for gm in GAMMAS])

### 4e. Inne wagi kanałów mastera (zastępstwo błędu czynników struktury)
Każdy kanał (pozycja atomu) ×0.8 i ×1.2. To przybliżenie — w teorii dynamicznej wkłady
pozycji nie są niezależne — ale daje realistyczny *rodzaj* błędu: zmienia względne
natężenia pasm, a nie geometrię.

In [ ]:
W_FACT = [0.8, 1.2]
n_ch = len(ch)
names_w = []
if n_ch > 1:
    for k in range(n_ch):
        for f in W_FACT:
            w = np.ones(n_ch); w[k] = f
            name = f"kanał {k} ×{f}"
            run(name, of.ConeModel(mm.reweighted_master(mp, MASTER, w), THETA, INVERT))
            names_w.append(name)
    display(summary(names_w))
else:
    print("master ma jeden kanał — pomijam")

### 4f. Inny master jako prawda (jeśli podany)

In [ ]:
if MASTER_TRUTH:
    mpt = ec.MasterPattern(MASTER_TRUTH)
    run(f"master: {os.path.basename(MASTER_TRUTH)}", of.ConeModel(mpt, THETA, INVERT))
    display(summary([f"master: {os.path.basename(MASTER_TRUTH)}"]))
else:
    print("MASTER_TRUTH = None — pomijam")

## 5. Przegląd: wybór wariantu i obciążenie orientacji

In [ ]:
names = list(results)
pc = np.array([results[k].poprawny.mean() for k in names])
bm = np.array([results[k].bias.median() for k in names])
rb = np.array([results[k].r_best.median() for k in names])
fig, ax = plt.subplots(1, 3, figsize=(15, 0.32 * len(names) + 1.5), sharey=True)
y = np.arange(len(names))
ax[0].barh(y, pc, color=np.where(pc > 0.99, "tab:green", np.where(pc > 0.9, "tab:orange", "tab:red")))
ax[0].set_xlim(0, 1); ax[0].set_xlabel("poprawny wariant")
ax[0].set_yticks(y); ax[0].set_yticklabels(names); ax[0].invert_yaxis()
ax[1].barh(y, bm); ax[1].set_xscale("log"); ax[1].set_xlabel("mediana biasu orientacji [°]")
ax[2].barh(y, 1 - rb); ax[2].set_xscale("log"); ax[2].set_xlabel("1 − r_best (mediana)")
for a in ax[1:]:
    a.axvline(1 - results[names[0]].r_best.median() if a is ax[2] else results[names[0]].bias.median(),
              color="k", ls=":", lw=1)
plt.tight_layout(); plt.show()

Prawy panel to rzecz, którą **zobaczysz na prawdziwych danych**: typowe 1 − r najlepszego
dopasowania. Porównując go z pomiarem możesz ocenić, który scenariusz (i jak silny) jest
realistyczny.

## 6. Czy margines Δr_obs mówi, kiedy ufać decyzji?

Łączymy wszystkie scenariusze i patrzymy na P(poprawny | Δr_obs). Jeśli istnieje próg Δr*,
powyżej którego decyzja jest poprawna *niezależnie od scenariusza*, to na pomiarze możesz
oznaczać piksele z pewnym wyborem osi c.

In [ ]:
allr = pd.concat({k: v for k, v in results.items()}, names=["scenariusz", "i"]).reset_index()
allr["rodzina"] = allr.scenariusz.str.split(" ").str[0]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.logspace(-5, -0.5, 40)
ax[0].hist(np.clip(allr.dr_obs[allr.poprawny], 1e-5, None), bins=bins, alpha=0.6, label="poprawny")
ax[0].hist(np.clip(allr.dr_obs[~allr.poprawny], 1e-5, None), bins=bins, alpha=0.8, label="błędny")
ax[0].set_xscale("log"); ax[0].set_xlabel("Δr_obs = r_1 − r_2"); ax[0].set_ylabel("liczba")
ax[0].legend()

edges = np.logspace(-5, -0.5, 16)
mid = np.sqrt(edges[1:] * edges[:-1])
for fam, d in list(allr.groupby("rodzina")) + [("WSZYSTKIE", allr)]:
    idx = np.digitize(d.dr_obs, edges) - 1
    acc = [d.poprawny[idx == b].mean() if (idx == b).sum() >= 5 else np.nan for b in range(len(mid))]
    ax[1].plot(mid, acc, "o-" if fam == "WSZYSTKIE" else ".-", lw=2.5 if fam == "WSZYSTKIE" else 1,
               color="k" if fam == "WSZYSTKIE" else None, label=fam)
ax[1].set_xscale("log"); ax[1].set_ylim(-0.02, 1.02)
ax[1].set_xlabel("Δr_obs"); ax[1].set_ylabel("P(poprawny wariant)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

# próg: najmniejsze Δr*, powyżej którego >= 99% poprawnych (łącznie)
d = allr.sort_values("dr_obs", ascending=False).reset_index(drop=True)
acc_above = d.poprawny.cumsum() / (np.arange(len(d)) + 1)
ok = np.where(acc_above >= 0.99)[0]
if len(ok):
    j = ok.max()
    print(f"Δr* ≈ {d.dr_obs[j]:.4f}: powyżej {acc_above[j]:.1%} poprawnych, "
          f"zostaje {(j + 1) / len(d):.0%} pikseli (łącznie ze wszystkich scenariuszy)")
    per = d[d.dr_obs >= d.dr_obs[j]].groupby("scenariusz").poprawny.agg(["mean", "size"])
    worst = per[per["mean"] < 1].sort_values("mean")
    print("scenariusze z błędami powyżej progu:" if len(worst) else "powyżej progu brak błędów w żadnym scenariuszu")
    if len(worst):
        print(worst.rename(columns={"mean": "poprawnych", "size": "pikseli"}).to_string())

In [ ]:
# ΔlogL zawyża pewność, gdy reszty nie są i.i.d. — porównanie: ile błędnych decyzji
# miało "bardzo pewne" ΔlogL (> 10, czyli nominalnie e^10 : 1)?
for k, v in results.items():
    bad = v[~v.poprawny]
    if len(bad):
        print(f"{k:<28} błędnych {len(bad):3d}, w tym z ΔlogL_obs > 10: {(bad.dlogL_obs > 10).sum():3d}"
              f"   (mediana ΔlogL błędnych {bad.dlogL_obs.median():.1f})")

## 7. Łączenie pikseli domeny

`N_DOM` domen, w każdej `K` pikseli o tej samej orientacji i niezależnym szumie.
Dla domeny sumujemy log-wiarygodności wariantów: log L_v = −n/2 · log(1 − r_v²).
Szum uśrednia się jak 1/√K; niedopasowanie strukturalne jest wspólne dla wszystkich pikseli
domeny, więc **błąd systematyczny się nie uśrednia** — może prowadzić do pewnej, ale złej decyzji.

In [ ]:
N_DOM, K_LIST = 20, [1, 3, 10, 30]
AGG_SCEN = ["ref: bez niedopasowania", "θ +0.5°", "σ_conv 0.5°", "γ 0.85"]   # klucze z results
g_dom = of.random_orientations(N_DOM, SEED + 100)
Kmax = max(K_LIST)
agg = {}
for name in AGG_SCEN:
    if name not in SCEN:
        continue
    t0 = time.perf_counter()
    truth, tr = SCEN[name]["truth"], SCEN[name]["transform"]
    Y = truth.profiles(g_dom, PHI0, SENSE)
    Y = tr(Y) if tr is not None else Y
    logL = np.zeros((N_DOM, Kmax, len(VARIANTS)))
    for d in range(N_DOM):
        Yd = perturb(np.repeat(Y[d:d + 1], Kmax, 0), seed=1000 + d)
        for k in range(Kmax):
            _, R = of.variant_scores(fit, Yd[k], g_dom[d], VARIANTS, PHI0, SENSE)
            logL[d, k] = -0.5 * N_OMEGA * np.log(np.clip(1 - R ** 2, 1e-300, None))
    # wariant 0 = prawdziwy (startowaliśmy z V g_true, V[0] = I)
    agg[name] = {K: np.mean(logL[:, :K].sum(1).argmax(1) == 0) for K in K_LIST}
    print(f"[{name}] {time.perf_counter() - t0:.0f} s")

pd.DataFrame(agg).rename_axis("K pikseli").style.format("{:.0%}")

Interpretacja: jeśli w scenariuszu z niedopasowaniem odsetek poprawnych domen **rośnie** z K,
błąd jest "szumopodobny" i agregacja pomaga. Jeśli **stoi albo maleje** (dąży do 0% albo 100%
w zależności od orientacji), niedopasowanie jest systematyczne: agregacja daje pewność,
ale nie poprawność — wtedy trzeba najpierw usunąć niedopasowanie (kalibracja θ, σ, γ).

## 8. Zestawienie i zapis

In [ ]:
display(summary())
pd.concat({k: v.drop(columns="R") for k, v in results.items()}, names=["scenariusz", "i"]) \
  .to_csv("mismatch_tests.csv")
print("zapisano mismatch_tests.csv")

## 9. Kalibracja θ z danych (bez etykiet)

θ jest wspólne dla całej mapy, więc można je wyznaczyć z samych profili:

  S(θ) = Σ_piksele max_wariant [ −n/2 · log(1 − r²) ],   θ̂ = argmax S.

Maksimum po wariantach oznacza, że **wybór osi c nie jest potrzebny do kalibracji** — procedura
działa tak samo na pomiarze. Przebieg (identyczny jak później na prawdziwych danych):

1. orientacje startowe G0: pełne dopasowanie (bank + warianty) przy **nominalnym** θ
   (`START_FROM = "full"`) — albo, dla porównania, prawdziwe orientacje (`"truth"`),
2. skan zgrubny θ ± 1.2° co 0.1°, potem dokładny ± 0.15° co 0.025° wokół maksimum,
   przy każdym θ dopracowanie 6 wariantów z ciepłego startu G0,
3. ocena na **nowym** zbiorze pikseli: poprawność wariantu i bias przy θ nominalnym, θ̂ i θ prawdziwym,
4. ile pikseli potrzeba: θ̂ z losowych podzbiorów K pikseli (bez ponownych obliczeń).

Scenariusze: `(Δθ, σ_conv prawdy)`. Ostatni sprawdza, czy inne niedopasowanie (rozmycie)
nie obciąża θ̂. Δθ = +0.17° celowo nie leży na siatce zgrubnej.

In [ ]:
import calibration as cal

CAL_SCEN   = [(+0.5, 0.0), (-0.3, 0.0), (+0.17, 0.5)]   # (Δθ prawdy, σ_conv prawdy)
N_CAL      = 30          # pikseli do kalibracji
N_EVAL     = 60          # nowych pikseli do oceny po kalibracji
START_FROM = "full"      # "full" (jak na pomiarze) albo "truth"
K_SUB      = [3, 5, 10, 20, 30]

if START_FROM == "full" and bank is None:     # bank przy NOMINALNYM θ
    path = f"{BANK_CACHE}_cone_sp{SPACING}_bg0.npz" if BANK_CACHE else None
    bank = of.Bank.load(path) if path and os.path.exists(path) else of.build_bank(fit, SPACING)
    if path and not os.path.exists(path):
        bank.save(path)


def eval_with(model, Y, G):
    # ocena jak w sekcji 2 (start z wariantów prawdy), ale dowolnym modelem
    rows = []
    for y, gt in zip(Y, G):
        Gv, R = of.variant_scores(model, y, gt, VARIANTS, PHI0, SENSE)
        mis = of.misorientation_deg(Gv, np.repeat(gt[None], len(Gv), 0))
        it = int(np.argmin(mis))
        rows.append(dict(poprawny=int(np.argmax(R)) == it, bias=mis[it], r_best=R.max()))
    return pd.DataFrame(rows)


cal_out = {}
for dth, sig in CAL_SCEN:
    name = f"Δθ {dth:+.2f}°, σ {sig}°"
    th_true = THETA + dth
    truth = of.ConeModel(mp, th_true, INVERT, sigma_conv_deg=sig)
    print(f"=== {name}: θ prawdziwe = {th_true:.3f}° ===")
    t0 = time.perf_counter()

    # --- dane kalibracyjne
    g_cal = of.random_orientations(N_CAL, SEED + 500)
    Y_cal = perturb(truth.profiles(g_cal, PHI0, SENSE), seed=501)

    # --- orientacje startowe przy nominalnym θ (bez wiedzy o prawdzie)
    if START_FROM == "full":
        G0 = np.stack([of.fit_profile(fit, bank, y, PHI0, SENSE, top_k=TOP_K, variants=VARIANTS,
                                      n_refine=N_REFINE)["g"] for y in Y_cal])
    else:
        G0 = g_cal.copy()
    print(f"  starty gotowe ({time.perf_counter() - t0:.0f} s)")

    # --- skan θ
    th_hat, scan = cal.calibrate_theta(mp, Y_cal, G0, THETA, phi0=PHI0, sense=SENSE,
                                       invert=INVERT, variants=VARIANTS)
    # który wariant (względem G0) jest prawdziwy — tylko do diagnostyki
    vtrue = np.array([np.argmin(of.misorientation_deg(VARIANTS @ G0[p],
                                                      np.repeat(g_cal[p][None], 6, 0)))
                      for p in range(N_CAL)])

    # --- ocena na nowych pikselach
    g_ev = of.random_orientations(N_EVAL, SEED + 600)
    Y_ev = perturb(truth.profiles(g_ev, PHI0, SENSE), seed=601)
    ev = {lab: eval_with(of.ConeModel(mp, th, INVERT), Y_ev, g_ev)
          for lab, th in [("nominalne", THETA), ("θ̂", th_hat), ("prawdziwe", th_true)]}

    # --- ile pikseli potrzeba
    rng = np.random.default_rng(7)
    sub = {K: np.array([cal.theta_hat(scan, rng.choice(N_CAL, K, replace=False))
                        for _ in range(200)]) for K in K_SUB if K <= N_CAL}

    cal_out[name] = dict(th_true=th_true, th_hat=th_hat, scan=scan, vtrue=vtrue, ev=ev, sub=sub)
    print(f"  θ̂ = {th_hat:.3f}° (błąd {th_hat - th_true:+.3f}°), "
          f"razem {time.perf_counter() - t0:.0f} s")

In [ ]:
rows = {}
for name, o in cal_out.items():
    r = {"θ prawdziwe": o["th_true"], "θ̂": o["th_hat"], "błąd θ̂ [°]": o["th_hat"] - o["th_true"]}
    for lab, df in o["ev"].items():
        r[f"poprawny ({lab})"] = df.poprawny.mean()
        r[f"bias med ({lab}) [°]"] = df.bias.median()
    rows[name] = r
fmt = {c: ("{:.0%}" if c.startswith("poprawny") else "{:.3f}") for c in next(iter(rows.values()))}
pd.DataFrame(rows).T.style.format(fmt)

In [ ]:
fig, ax = plt.subplots(len(cal_out), 3, figsize=(15, 3.6 * len(cal_out)), squeeze=False)
for row, (name, o) in zip(ax, cal_out.items()):
    sc, th = o["scan"], o["scan"]["thetas"]
    # S(θ) w jednostkach średniego r_best (czytelniejsze niż suma logL)
    rbest = sc["R"].max(2).mean(1)
    row[0].plot(th, rbest, ".-")
    row[0].axvline(o["th_true"], color="k", ls=":", label="prawda")
    row[0].axvline(o["th_hat"], color="tab:red", ls="--", label="θ̂")
    row[0].axvline(THETA, color="0.6", ls="-.", label="nominalne")
    row[0].set_xlabel("θ modelu [°]"); row[0].set_ylabel("średnie r_best"); row[0].set_title(name)
    row[0].legend(fontsize=8)
    # poprawność wariantu w funkcji θ (diagnostyka; na pomiarze niewidoczna)
    acc = (sc["vbest"] == o["vtrue"][None, :]).mean(1)
    row[1].plot(th, acc, ".-")
    row[1].axvline(o["th_true"], color="k", ls=":"); row[1].axvline(o["th_hat"], color="tab:red", ls="--")
    row[1].set_ylim(-0.02, 1.02); row[1].set_xlabel("θ modelu [°]")
    row[1].set_ylabel("poprawny wariant (piksele kalibr.)")
    # rozrzut θ̂ z K pikseli
    Ks = list(o["sub"])
    row[2].boxplot([o["sub"][K] - o["th_true"] for K in Ks])
    row[2].set_xticks(range(1, len(Ks) + 1)); row[2].set_xticklabels([str(K) for K in Ks])
    row[2].axhline(0, color="k", lw=0.8)
    row[2].set_xlabel("K pikseli w kalibracji"); row[2].set_ylabel("θ̂ − θ_prawdziwe [°]")
    row[2].set_title("ile pikseli potrzeba?")
plt.tight_layout(); plt.show()

for name, o in cal_out.items():
    print(name, " ".join(f"K={K}: sd={o['sub'][K].std():.3f}°" for K in o["sub"]))

**Jak czytać:**

* lewy panel — kształt S(θ). Ostre maksimum = θ dobrze wyznaczalne. To samo policzysz na pomiarze;
  szerokość piku (spadek r o np. 0.01) daje skalę dokładności kalibracji,
* środkowy — jak szybko psuje się wybór osi c, gdy θ odchodzi od prawdy. Szerokość "plateau"
  wokół prawdy to wymagana dokładność θ; porównaj ją z rozrzutem θ̂ z prawego panelu,
* prawy — rozrzut θ̂ z K pikseli: jeśli już kilka pikseli daje rozrzut ≪ szerokości plateau,
  kalibracja nie jest wąskim gardłem,
* scenariusz z σ_conv: jeśli θ̂ jest przesunięte względem prawdy, inne niedopasowania
  obciążają kalibrację — wtedy kalibrować θ i σ łącznie.

**Na pomiarze:** `cal.calibrate_theta(mp, Y_pomiar, G0, 40.0, phi0=0, sense=..., mask=..., n_bg_harm=3)`,
z G0 z `fit_profile` przy θ = 40° (albo z metody słownikowej, jeśli φ0 jest już znane).
Najlepiej na pikselach z dala od artefaktów i nasycenia; z prawego panelu wynika, ilu potrzeba.